# Criba de motores TTS — dots.tts (SOAR)

`dots-studio/dots.tts-soar`, el checkpoint de mayor similitud y base de su fine-tuning, en modo continuación (referencia + su transcripción), `language="es"`, `num_steps=10`, `guidance_scale=1.2`.

dots.tts solo admite Python 3.10–3.12 y Colab ya trae 3.13: la celda 1 crea un Python 3.12 aparte (con `uv`) solo para él, y la celda 3 genera desde ahí con `criba/motores/dots_generar.py`.

**GPU:** L4. **Tiempo estimado:** unos 45 min, incluida la instalación.

**Cómo se usa** (es igual en todos los cuadernos de la criba):

1. *Entorno de ejecución → Cambiar tipo de entorno* → **L4**.
2. Ejecuta la **celda 1** (instalación). Cuando termine, *Entorno de ejecución → Reiniciar sesión*.
3. Ejecuta las demás celdas **en orden**, de la 2 a la última. La celda 2 comprueba que estén las
   referencias y las frases antes de cargar ningún modelo: si dice `RESULTADO: REVISAR`, para ahí.
4. La última celda **desconecta la GPU sola** para que no siga gastando unidades.

Si la sesión se corta a medias, vuelve a empezar por la celda 1: lo que ya esté en Drive no se
genera otra vez.

Salida: `fine_tunned_naval/outputs/criba/<variante>/` en Drive, un `.wav` por item (`n01`, `v01`,
`b01`, `k01`…) más `registro.jsonl`, `parametros.json` y `entorno.json`.

In [ ]:
# 1. Instalación: un Python 3.12 aparte solo para dots.tts (unos 5 min)
# Aquí no hace falta reiniciar la sesión, pero no pasa nada si lo haces.
!pip install -q uv
!uv python install 3.12
!uv venv -q --python 3.12 /content/venv_dots
# torch y torchaudio tienen que ir a juego: se fijan las versiones que recomienda dots.tts
# (constraints/recommended.txt). transformers 4.57.0 está retirada de PyPI: se usa la 4.57.1.
!uv pip install -q --python /content/venv_dots/bin/python dots-tts==0.3.1 torch==2.8.0 torchaudio==2.8.0 transformers==4.57.1 soundfile numpy
!/content/venv_dots/bin/python -c "import dots_tts, torch; print('dots-tts OK | torch', torch.__version__, '| CUDA', torch.cuda.is_available())"

In [ ]:
# 2. Preparar: código del repo, Drive, frases y referencias
import os, sys, subprocess
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun

print(sys.version.split()[0], "|", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                                  capture_output=True, text=True).stdout.strip())
comun.montar_drive()
ITEMS, SILENCIO_MS = comun.cargar_items(None)
REFS = comun.comprobar(ITEMS)

In [ ]:
# 3. Generar (en el Python 3.12 aparte; se salta lo que ya esté en Drive)
!/content/venv_dots/bin/python -u /content/repo/criba/motores/dots_generar.py

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
for variante in ['dots_soar']:
    reg = comun.SALIDA / variante / "registro.jsonl"
    if not reg.exists():
        print(f"{variante}: sin registro")
        continue
    filas = [json.loads(l) for l in reg.read_text(encoding="utf-8").splitlines()]
    ok = sum(f["ok"] for f in filas)
    print(f"{variante}: {ok} items correctos, {len(filas) - ok} con error")
comun.desconectar()